In [1]:
!nvidia-smi

import torch
torch.cuda.is_available()

Tue Dec  9 11:54:26 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.129.03             Driver Version: 535.129.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla V100-PCIE-32GB           Off | 00000000:D8:00.0 Off |                    0 |
| N/A   30C    P0              26W / 250W |      0MiB / 32768MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

True

In [2]:
!pip install -U torch transformers datasets scikit-learn pandas
!pip install "accelerate>=0.26.0"




Defaulting to user installation because normal site-packages is not writeable


Defaulting to user installation because normal site-packages is not writeable


In [3]:
import pandas as pd

human_path = "election2024_tweets.csv"
ai_path    = "synthetic_election2024.csv"

human_df = pd.read_csv(human_path)
ai_df    = pd.read_csv(ai_path)

print("Human columns:", human_df.columns)
print("AI columns:", ai_df.columns)

human_df.head()


Human columns: Index(['Tweet ID', 'Tweet_count', 'Username', 'Text', 'Created At', 'Retweets',
       'Likes', 'User Bio', 'Follower Count', 'Following Count', 'Replies',
       'Location', 'Age', 'Gender', 'Hashtags'],
      dtype='object')
AI columns: Index(['Text'], dtype='object')


,Tweet ID,Tweet_count,Username,Text,Created At,Retweets,Likes,User Bio,Follower Count,Following Count,Replies,Location,Age,Gender,Hashtags
0,1857211912523850191,1,Andrew Olson (PCO) ⚔️⚔️⚔️,"""Trump stole it"", says dollar-store Obama.\n\n...",Thu Nov 14 23:59:59 +0000 2024,0,1,🇺🇸🍊⚔\nConservative. Former Dem who broke the ...,1350,1084,0,United States,Unknown,Unknown,[]
1,1857211911802114113,2,Sid,@SpeakerJohnson @SenJohnThune I think you need...,Thu Nov 14 23:59:59 +0000 2024,0,0,NaN,37,88,0,Unknown,Unknown,Unknown,[]
2,1857211911713988946,3,🇺🇸 Alf 🇺🇸,@EricLDaugh Insane the 10 GOP Senators all vot...,Thu Nov 14 23:59:59 +0000 2024,0,1,RIP Charlie Kirk 🙏🏻🇺🇸,2386,1664,0,"MAGA, USA 🇺🇸",Unknown,Unknown,[]
3,1857211911626187170,4,Kathy,@TristanSnell 1- This is like his 50th federal...,Thu Nov 14 23:59:59 +0000 2024,15,51,"I love Big Brother, Hockey and to give my 2 cents",13401,12274,20,"Montreal, QC, Canada",Unknown,Unknown,[]
4,1857211911567233305,5,National Catholic Register,What the @USCCB Meeting Said About the US Bish...,Thu Nov 14 23:59:59 +0000 2024,0,4,"Our mission is to inform, inspire, challenge a...",209892,727,0,United States,Unknown,Unknown,[]


In [4]:
TEXT_COL = "Text"

In [5]:
# keep only the text column
human_df = human_df[[TEXT_COL]].copy()
ai_df    = ai_df[[TEXT_COL]].copy()

# label: 0 = scraped/human, 1 = AI
human_df["label"] = 0
ai_df["label"]    = 1

df = pd.concat([human_df, ai_df], ignore_index=True)

# basic cleaning
df = df.dropna(subset=[TEXT_COL])
df = df.drop_duplicates(subset=[TEXT_COL])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df.head(), df["label"].value_counts()


(                                                Text  label
 0  Breaking: Charlie Kirk hates freedom!!! Absolu...      1
 1  Consider how rediculous this looks! Donald Tru...      0
 2  Watching The GOP is fueling hate is so strawma...      1
 3                      @Pinovibes Pino trump is back      0
 4  Trump's big MAGA tent has too many holes in it...      0,
 label
 1    100935
 0     99501
 Name: count, dtype: int64)

In [6]:
from datasets import Dataset
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["label"]
)

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df.reset_index(drop=True))
len(train_ds), len(test_ds)


/home/gg2713/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(160348, 40088)

In [7]:
from transformers import AutoTokenizer

model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_batch(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

train_tok = train_ds.map(tokenize_batch, batched=True)
test_tok  = test_ds.map(tokenize_batch, batched=True)

# rename 'label' -> 'labels' and set format
train_tok = train_tok.rename_column("label", "labels")
test_tok  = test_tok.rename_column("label", "labels")

train_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])



Map: 100%|██████████| 40088/40088 [00:03<00:00, 11104.40 examples/s]


In [9]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

model.to("cuda")


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [11]:
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
    }

training_args = TrainingArguments(
    output_dir="./roberta_elec2024",
    eval_strategy="epoch",         # old HuggingFace API
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    compute_metrics=compute_metrics,
)

trainer.train()


Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.000000,0.001596,0.999850,0.999703,1.000000,0.999851
2,0.000000,0.001625,0.999850,0.999703,1.000000,0.999851
3,0.000000,0.002230,0.999775,0.999554,1.000000,0.999777


TrainOutput(global_step=60132, training_loss=0.003805469387660651, metrics={'train_runtime': 4446.0536, 'train_samples_per_second': 108.196, 'train_steps_per_second': 13.525, 'total_flos': 3.164199862864896e+16, 'train_loss': 0.003805469387660651, 'epoch': 3.0})

In [19]:
metrics = trainer.evaluate()
metrics


{'eval_loss': 0.002229937119409442,
 'eval_accuracy': 0.9997754939133905,
 'eval_precision': 0.999554367201426,
 'eval_recall': 1.0,
 'eval_f1': 0.9997771339425006,
 'eval_runtime': 100.072,
 'eval_samples_per_second': 400.591,
 'eval_steps_per_second': 50.074,
 'epoch': 3.0}

In [12]:
trainer.save_model("./roberta_elec2024/best_model")
tokenizer.save_pretrained("./roberta_elec2024/best_model")


('./roberta_elec2024/best_model/tokenizer_config.json',
 './roberta_elec2024/best_model/special_tokens_map.json',
 './roberta_elec2024/best_model/vocab.json',
 './roberta_elec2024/best_model/merges.txt',
 './roberta_elec2024/best_model/added_tokens.json',
 './roberta_elec2024/best_model/tokenizer.json')

In [20]:
import pandas as pd
from datasets import Dataset

# Update this if the file name is different
prellm_path = "midterm_tweets221.csv"

pre_df = pd.read_csv(prellm_path)
print(pre_df.columns)
pre_df.head()


Index(['Tweet ID', 'Tweet_count', 'Username', 'Text', 'Created At', 'Retweets',
       'Likes', 'User Bio', 'Follower Count', 'Following Count', 'Replies',
       'Location', 'Age', 'Gender', 'Hashtags'],
      dtype='object')


,Tweet ID,Tweet_count,Username,Text,Created At,Retweets,Likes,User Bio,Follower Count,Following Count,Replies,Location,Age,Gender,Hashtags
0,1597742174804013056,1,Joe Duffany 🇺🇸,@Brye325 @julie_kelly2 Hilarious from a vet wh...,Tue Nov 29 23:59:53 +0000 2022,0,0,2021 Model 3 Performance | 2025 Model Y Perfor...,57,417,1,Unknown,Unknown,Unknown,"['loser', 'maga']"
1,1597742171456618498,2,D. Staples,'Anti-woke' MAGA bank is shutting down after m...,Tue Nov 29 23:59:52 +0000 2022,0,0,NaN,247,29,0,Unknown,Unknown,Unknown,"['Conservatives', 'Republicans', 'MAGA', 'edGl..."
2,1597742124229079040,3,Resisting PerSister - done with weapons of WAR,#GOP and #MAGAts like send out tweets about Bi...,Tue Nov 29 23:59:41 +0000 2022,4,5,Retired elected municipal official & FEMA cont...,8899,9876,1,Cape Cod & I love this group:,Unknown,Unknown,"['GOP', 'MAGAts']"
3,1597742108135522304,4,Happy...Loved...GRATEFUL!,"What #democRATS' motto should be is ""don't bel...",Tue Nov 29 23:59:37 +0000 2022,0,1,"❤️GOD, my HUSBAND, my ANIMALS..& TRUMP! DOGS ...",3539,3762,0,United States,Unknown,Unknown,"['democRATS', '1', 'democRAT', 'lies', 'Georgi..."
4,1597741952073871363,5,JC Yoo,@JoJoFromJerz And who do we have to thank? Ir...,Tue Nov 29 23:59:00 +0000 2022,0,0,Ipsa Scientia Potestas Est,90,156,0,"Berkeley, CA",Unknown,Unknown,"['GOP', 'USAvsIran', 'FIFA23']"


In [21]:
TEXT_COL = "Text"   # ← change this to match your actual text column


In [22]:
pre_df = pre_df[[TEXT_COL]].dropna().copy()
pre_df["label"] = 0   # all are true human by construction

len(pre_df), pre_df.head()


(31227,
                                                 Text  label
 0  @Brye325 @julie_kelly2 Hilarious from a vet wh...      0
 1  'Anti-woke' MAGA bank is shutting down after m...      0
 2  #GOP and #MAGAts like send out tweets about Bi...      0
 3  What #democRATS' motto should be is "don't bel...      0
 4  @JoJoFromJerz And who do we have to thank?  Ir...      0)

In [23]:
pre_ds = Dataset.from_pandas(pre_df.reset_index(drop=True))

pre_tok = pre_ds.map(tokenize_batch, batched=True)
pre_tok = pre_tok.rename_column("label", "labels")
pre_tok.set_format("torch", columns=["input_ids", "attention_mask", "labels"])


Map: 100%|██████████| 31227/31227 [00:03<00:00, 9505.26 examples/s] 


In [25]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

pre_preds = trainer.predict(pre_tok)

y_true_pre = pre_preds.label_ids          # all zeros
y_pred_pre = np.argmax(pre_preds.predictions, axis=-1)

print("Classification report on pre-LLM tweets:")
print(classification_report(y_true_pre, y_pred_pre))

print("Confusion matrix:")
print(confusion_matrix(y_true_pre, y_pred_pre))



/home/gg2713/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Classification report on pre-LLM tweets:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     31227
           1       0.00      0.00      0.00         0

    accuracy                           1.00     31227
   macro avg       0.50      0.50      0.50     31227
weighted avg       1.00      1.00      1.00     31227

Confusion matrix:
[[31159    68]
 [    0     0]]


/home/gg2713/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/gg2713/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/gg2713/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [26]:
import numpy as np
from sklearn.metrics import classification_report

# Predict on test set
preds = trainer.predict(test_tok)
y_true = preds.label_ids
y_pred = np.argmax(preds.predictions, axis=-1)

# test_df still corresponds to test_tok rows
sources = test_df["source"].values

print("Overall:")
print(classification_report(y_true, y_pred))

for src in ["human_scraped", "ai_clean", "ai_adv"]:
    mask = (sources == src)
    print(f"\nSource = {src}")
    print(classification_report(y_true[mask], y_pred[mask]))


KeyError: 'source'